In [4]:
from glob import glob
from PIL import Image
import os
import sys
from tqdm import tqdm
import subprocess
import json
from urllib.parse import quote
import multiprocessing
import pytesseract
import numpy as np
import cv2
import matplotlib.pyplot as plt

In [3]:
src = './portfolio_alias'

In [ ]:
# from pathlib import Path

# def replace_underscores_with_spaces(target_folder):
#     folder = Path(target_folder)
    
#     # .rglob("*") matches all files and subfolders recursively
#     # We wrap it in a list so we capture the file states BEFORE renaming anything
#     for file_path in list(folder.rglob("*")):
        
#         # Only rename files, leave subfolder names alone
#         if file_path.is_file():
#             current_name = file_path.name
            
#             if "_" in current_name:
#                 # Create the new name with spaces
#                 new_name = current_name.replace("_", " ")
                
#                 # Create the full new path in the same directory
#                 new_path = file_path.with_name(new_name)
                
#                 # Rename the file in place
#                 file_path.rename(new_path)
#                 print(f"Renamed: {current_name} -> {new_name}")

# replace_underscores_with_spaces(src)

In [ ]:
# import os
# import shutil

# if os.path.exists(dest):
#     if os.path.exists(dest + "_backup"):
#         shutil.rmtree(dest + "_backup")
#     shutil.copytree(dest, dest + "_backup")
#     shutil.rmtree(dest)
# os.makedirs(dest)

KeyboardInterrupt: 

In [8]:
portfolioFiles = [f for f in os.listdir(src) if not f[0] == "." and f.endswith(".jpg")]

portfolioFiles

['Left Turns Only.jpg',
 'Church of San Giorgio Maggiore, Facade.jpg',
 'Guggenheim Crowd.jpg',
 'Pescadero Migration.jpg',
 'West Shore Sunset.jpg',
 'Light Painting, Halo.jpg',
 'Pescadero Wavefront.jpg',
 'Barcelona Rooftops, Night.jpg',
 'Through the Oculus.jpg',
 'Stained Glass.jpg',
 'El Yunque Canopies.jpg',
 'Manhattan Bridge.jpg',
 'Midtown Divide.jpg',
 'Anchors of Venice.jpg',
 'Little Grand Canyon.jpg',
 'University Arches.jpg',
 'Arches of New York City Hall.jpg',
 'Tech Tower.jpg',
 'Study Hours.jpg',
 'Gondola Row.jpg',
 'Les Angles Tableau.jpg',
 'Serenity of Lake Tahoe.jpg',
 'Acres for Grazing.jpg',
 'Brandi at Rest.jpg',
 'Keys Sanctuary.jpg',
 "Pont d'Avignon, Sunset.jpg",
 'Poipu Sundown.jpg',
 'Squaw Powder Awakening.jpg',
 'Squaw Frontier.jpg',
 'The Winter Garden.jpg',
 'Mornings of San Marco.jpg',
 'Venetian Shadows.jpg',
 'Thunderbird Formation.jpg',
 'Apple Steps.jpg',
 'Colosseum Structure.jpg',
 'Les Baux-de-Provence Masonry.jpg',
 'Mountain Mobile.jpg',
 '

In [ ]:
# for filename in tqdm(portfolioFiles):
#     shutil.copy(src + "/" + filename, dest + "/" + filename)

100%|██████████| 206/206 [06:43<00:00,  1.96s/it]


In [4]:
def exiftoolFile(filename):
    filepath = src + '/' + filename
    meta = subprocess.check_output([f'exiftool', filepath]).decode('utf-8')
    # escapedFilename = filename.replace(" ", "\\ ")
    # escapedSrc = src.replace(" ", "\\ ")
    # escapedDest = dest.replace(" ", "\\ ")
    # cmd = f'exiftool -overwrite_original -TagsFromFile {escapedSrc}/{escapedFilename} -all:all>all:all {escapedDest}/{escapedFilename}'
    # print(cmd)
    # os.system(cmd)
    md = {}
    meta.split('\n')
    for line in meta.split('\n'):
        parts = line.split(' : ')
        k = parts[0].strip()
        v = ' : '.join(parts[1:]).strip()
        md[k] = v
    return md

metaData = {}
for filename in tqdm(portfolioFiles):
    metaData[filename] = exiftoolFile(filename)

with open(src + "/metadata.json", "w") as f:
    json.dump(metaData, f, indent=4)
    
metaData

100%|██████████| 206/206 [00:14<00:00, 14.19it/s]


{'Left Turns Only.jpg': {'ExifTool Version Number': '13.55',
  'File Name': 'Left Turns Only.jpg',
  'Directory': './portfolio_alias',
  'File Size': '2.6 MB',
  'File Modification Date/Time': '2023:05:06 15:44:06-05:00',
  'File Access Date/Time': '2026:06:10 15:47:53-05:00',
  'File Inode Change Date/Time': '2026:06:10 15:47:51-05:00',
  'File Permissions': '-rw-------',
  'File Type': 'JPEG',
  'File Type Extension': 'jpg',
  'MIME Type': 'image/jpeg',
  'JFIF Version': '1.01',
  'Exif Byte Order': 'Big-endian (Motorola, MM)',
  'Make': 'NIKON CORPORATION',
  'Camera Model Name': 'NIKON D3200',
  'Orientation': 'Horizontal (normal)',
  'X Resolution': '300',
  'Y Resolution': '300',
  'Resolution Unit': 'inches',
  'Software': 'Ver.1.03',
  'Modify Date': '2017:12:16 20:21:40.10',
  'Reference Black White': '0 255 0 255 0 255',
  'Exposure Time': '10',
  'F Number': '5.0',
  'Exposure Program': 'Manual',
  'ISO': '200',
  'Sensitivity Type': 'Recommended Exposure Index',
  'Create D

In [5]:
from PIL import Image

def load_images_for_batch(filenames, base_dir):
    batch_images = []
    for filename in tqdm(filenames):
        # 1. Open the image file (lazy loading, doesn't load pixels yet)
        img = Image.open(f"{base_dir}/{filename}")
        
        # 2. Convert to RGB to avoid issues with PNG transparency (RGBA) or Grayscale
        if img.mode != "RGB":
            img = img.convert("RGB")
            
        # 3. Resize to a manageable size immediately to save RAM
        # CLIP ultimately uses 224x224, so resizing to 448x448 keeps plenty of detail
        img.thumbnail((448, 448))
        
        # 4. Force PIL to load the pixels into memory at this small size
        img.load() 
        
        batch_images.append(img)
    return batch_images

In [6]:
from transformers import CLIPProcessor, CLIPModel
import torch

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
print('loaded models')

device = torch.device('mps') if torch.backends.mps.is_available() else torch.device('cpu')
print(f'Using device: {device}')
model = model.to(device)


/Users/bsanie/GitHub/blakesanie.com/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 2334.24it/s, Materializing param=visual_projection.weight]                                
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking

loaded models
Using device: mps


In [9]:
# Configuration
BATCH_SIZE = 32  # Adjust based on your GPU memory
embeddingsPath = src + '/embeddings.json'
out = {}

# Process in batches
for i in range(0, len(portfolioFiles), BATCH_SIZE):
    batch_filenames = portfolioFiles[i : i + BATCH_SIZE]
    print(f"Processing batch {i // BATCH_SIZE + 1}/{(len(portfolioFiles) - 1) // BATCH_SIZE + 1}...")
    
    # 1. Load only the images needed for this batch
    images = load_images_for_batch(batch_filenames, src)
    print("loade images")
    # 2. Preprocess and move to device
    inputs = processor(images=images, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # 3. Model Inference
    with torch.no_grad():
        imageEmbeddings = model.get_image_features(**inputs)
    
    # 4. Handle different model output structures
    if hasattr(imageEmbeddings, 'image_embeds'):
        imageEmbeddings = imageEmbeddings.image_embeds
    elif hasattr(imageEmbeddings, 'pooler_output'):
        imageEmbeddings = imageEmbeddings.pooler_output
        
    # 5. L2 Normalize the embeddings
    imageEmbeddings /= torch.norm(imageEmbeddings, p=2, dim=-1, keepdim=True)
    
    # 6. Convert batch embeddings to standard Python list
    batch_embeddings = imageEmbeddings.cpu().detach().numpy().tolist()
    
    # 7. Map embeddings back to their respective filenames
    for j, filename in enumerate(batch_filenames):
        out[filename] = [x for x in batch_embeddings[j]] # float("{:0.4f}".format(x))
        
    # Optional: Clear CUDA cache per batch if memory is extremely tight
    # torch.cuda.empty_cache()

# Save final aggregated dictionary to JSON
embContents = json.dumps(out)
with open(embeddingsPath, "w") as outfile:
    outfile.write(embContents)
    print('Wrote all embeddings to ', embeddingsPath)

# Final cleanup
if torch.cuda.is_available():
    torch.cuda.empty_cache()

out, len(out)

Processing batch 1/7...


100%|██████████| 32/32 [00:03<00:00,  9.69it/s]


loade images
Processing batch 2/7...


100%|██████████| 32/32 [00:02<00:00, 12.40it/s]


loade images
Processing batch 3/7...


...

100%|██████████| 32/32 [00:02<00:00, 11.73it/s]


loade images
Processing batch 7/7...


100%|██████████| 14/14 [00:01<00:00, 10.66it/s]


loade images
Wrote all embeddings to  ./portfolio_alias/embeddings.json


({'Left Turns Only.jpg': [-0.015315252356231213,
   0.04089903086423874,
   -0.015926983207464218,
   0.005567430518567562,
   0.001302311778999865,
   0.01774664781987667,
   0.040143270045518875,
   0.02664584293961525,
   -0.0019133451860398054,
   -0.011837506666779518,
   0.006721384823322296,
   -0.005170885007828474,
   0.017661668360233307,
   0.010511637665331364,
   0.008169841021299362,
   -0.03533216938376427,
   -0.1344384104013443,
   0.01924259029328823,
   -0.031849972903728485,
   -0.02619178220629692,
   0.009412907995283604,
   -0.015024998225271702,
   -0.04665651544928551,
   0.0008302059723064303,
   0.0028583593666553497,
   -0.02891390398144722,
   0.008001970127224922,
   0.008122705854475498,
   -0.00281599466688931,
   -0.013721468858420849,
   -0.012644916772842407,
   0.025862541049718857,
   -0.01582496240735054,
   -0.007354683708399534,
   0.018248647451400757,
   -0.003862244775518775,
   -0.014792156405746937,
   0.01817741058766842,
   0.0253261793404

In [7]:
portfolioFiles

['Left Turns Only.jpg',
 'Church of San Giorgio Maggiore, Facade.jpg',
 'Guggenheim Crowd.jpg',
 'Pescadero Migration.jpg',
 'West Shore Sunset.jpg',
 'Light Painting, Halo.jpg',
 'Pescadero Wavefront.jpg',
 'Barcelona Rooftops, Night.jpg',
 'Through the Oculus.jpg',
 'Stained Glass.jpg',
 'El Yunque Canopies.jpg',
 'Manhattan Bridge.jpg',
 'Midtown Divide.jpg',
 'Anchors of Venice.jpg',
 'Little Grand Canyon.jpg',
 'University Arches.jpg',
 'Arches of New York City Hall.jpg',
 'Tech Tower.jpg',
 'Study Hours.jpg',
 'Gondola Row.jpg',
 'Les Angles Tableau.jpg',
 'Serenity of Lake Tahoe.jpg',
 'Acres for Grazing.jpg',
 'Brandi at Rest.jpg',
 'Keys Sanctuary.jpg',
 "Pont d'Avignon, Sunset.jpg",
 'Poipu Sundown.jpg',
 'Squaw Powder Awakening.jpg',
 'Squaw Frontier.jpg',
 'The Winter Garden.jpg',
 'Mornings of San Marco.jpg',
 'Venetian Shadows.jpg',
 'Thunderbird Formation.jpg',
 'Apple Steps.jpg',
 'Colosseum Structure.jpg',
 'Les Baux-de-Provence Masonry.jpg',
 'Mountain Mobile.jpg',
 '